source: https://www.bea.gov/data/gdp/gross-domestic-product

Creating Vintage Time Series of BEA estimations and revisions of US GDP.

In [ ]:
import numpy as np
# import polars as pl
import pandas as pd
# import os
# import glob
import re
# from pprint import pprint
# import matplotlib.pyplot as plt
# from matplotlib.animation import FuncAnimation
import sys 

In [ ]:
# Publication Date and readme info
# --- Extract text info (first 5 rows from Vintage History) ---
text_info_df = pd.read_excel(
    'gdp-gdi-vintage-history.xlsx', 
    sheet_name='Vintage History',
    header=None,
    nrows=4)

# Get first 5 rows and convert everything to string
text_info_df = text_info_df.astype(str)
text_info_df = text_info_df.fillna("")

# Flatten into a list of strings (optional)
text_info_strings = text_info_df.values.flatten().tolist()

# Remove 'nan' strings if needed
text_info_strings = [s for s in text_info_strings if s.lower() != 'nan']

for item in text_info_strings:
    print(item)

In [ ]:
# --- Extract read_me sheet (Sheet 2) ---
readme = pd.read_excel( 'gdp-gdi-vintage-history.xlsx', 
    sheet_name='ReadMe', 
    header=None)

# Convert all to strings
readme_strings = readme.astype(str)

# Flatten 
readme_list = readme_strings.values.flatten().tolist()

# Remove empty/nan
readme_list = [s for s in readme_list if s.lower() != 'nan']

for item in readme_list:
    print(item)


In [ ]:
# Download current version of the excel file from
# https://www.bea.gov/sites/default/files/2026-02/gdp-gdi-vintage-history.xlsx
# read excel file from sheet 'Vintage History' into data frame 'raw'
raw  = pd.read_excel('gdp-gdi-vintage-history.xlsx', sheet_name='Vintage History',
    skiprows=5,
    header=None)

Table needs to be cleaned:
- each quarter with its' estimations and revisions is listed in an individual table
    - Identify boundries of the tables using column (0)
- Create meaningfull headers

In [ ]:
# For each Quarter there is an individual table
# Fill nan in first column with new column to create quarter date
# New column name
raw['Quarter'] = raw[0]
# Down fill 'Quarter'
raw['Quarter'] = raw['Quarter'].ffill()

In [ ]:
# Transfer to Dataframe 
df = raw[[ 'Quarter', 1, 2, 3, 4, 5, 6]].copy()

In [ ]:
# Assign column names
df.columns = [
    "Quarter",
    "Type",
    "GDP",
    "GDI",
    "GDP_%_Change",
    "GDI_%_Change",
    "Release_Date"
]

In [ ]:
# Split Release_Date into Release_Date and Note 
df['Release_Date'] = df['Release_Date'].astype(str)
df['Note'] = df['Release_Date'].str[12:].str.strip().replace("",pd.NA)
df['Release_Date'] = df['Release_Date'].str[:12]

In [ ]:
# Change Release_Date into Date time
df['Release_Date'] = pd.to_datetime(
    df['Release_Date'],
    format="%b %d, %Y",
    errors='coerce'
)

In [ ]:
# Drop NaT Release_Date. These Rows are not needed
df = df.dropna(subset=['Release_Date'])

In [ ]:
# Change Data types of 'GDP','GDI','GDP_%_Change','GDI_%_Change'
cols = ['GDP','GDI','GDP_%_Change','GDI_%_Change']
df[cols] = (
    df[cols]
    .replace(".....", pd.NA) # handle missing marker
    .replace(",", "", regex=True)     # remove thousands separators
    .apply(pd.to_numeric, errors="coerce")
)

In [ ]:
# Create Quarter_Date column from Quarter.  
df['Quarter_Date'] = pd.PeriodIndex(df['Quarter'], freq='Q').to_timestamp(how='end').normalize()

In [ ]:
# Reset Index
df = df.reset_index(drop=True)

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Latest Release_Date for each Quarter
df['Last_Release_Date'] = (
    df.groupby('Quarter')['Release_Date'].transform('max')
)

In [ ]:
# Create Valid_From and Valid_To
# First check if df is sorted via 'Quarter_Date', 'Release_Date'
# If is_sorted is True then Valid_From and Valid_To can be created
is_sorted = (
    df.groupby('Quarter_Date')['Release_Date']
    .apply(lambda s: s.is_monotonic_decreasing)
    .all()
)
is_sorted	   

In [ ]:
# Extra Check for possible quaters that are not sorted if is_sorted is False 
bad_quarters = (
    df.groupby("Quarter")["Release_Date"]
      .apply(lambda s: not s.is_monotonic_decreasing)
)
for item in bad_quarters:
    print(item)

In [ ]:
# Create Valid_From and Valid_To
df['Valid_From'] = df['Release_Date']
df['Valid_To'] = (df.groupby('Quarter')['Release_Date'].shift(+1))

In [ ]:
# Verify Validity Intervals Per Quarter
test = (
    df.sort_values(['Quarter_Date', 'Release_Date'])
      .groupby('Quarter')
      .apply(lambda x: (x['Valid_From'].shift(-1) < x['Valid_To']).any())
)
print(test[test])

In [ ]:
df[['Quarter_Date', 'GDP']].loc[df['Release_Date'] == '2023-09-28']

In [ ]:
df.dtypes

In [ ]:
df.head(20)

In [ ]:
# Build time series using validity intervals
# Use row where as_of >= Valid_From and (as_of <= Valid_To OR Valid_To is NaT)
def build_vintage_timeseries(df):
    value_cols = [
        'GDP',
        'GDI',
        'GDP_%_Change',
        'GDI_%_Change'
    ]

    # List of Unique Release_Date
    release_dates = (
        df['Release_Date']
        .dropna()
        .sort_values()
        .unique()
    )

    results ={}

    # Filling Results
    for as_of in release_dates:
        snapsh = df[
            (df['Valid_From'] <= as_of) &
            (
                (df['Valid_To'] > as_of) |
                df['Valid_To'].isna()
                  )
        ]

        snapsh = (
            snapsh[
                ['Quarter', 'Quarter_Date'] + value_cols
            ]
            .sort_values('Quarter_Date')
            .reset_index(drop=True)
        )

        results[pd.Timestamp(as_of)] = snapsh

    return results

In [ ]:
# The time series are transfered as data frames into a dictionary
vint = build_vintage_timeseries(df)

In [ ]:
vint.keys()

In [ ]:
# Alternatively transfer of the time series into single data frame 
panel = (
    pd.concat(vint, names=["As_Of_Release_Date"])
      .reset_index(level=0)
)

In [ ]:
# Aggregated Results of Quarter estimations/calculations
df_stat = df.groupby('Quarter')[cols].agg(['min','max','mean','std'])

df_stat.head(10)

In [ ]:
# Calculate GDP % Change Switch
df_stat['GDP_%_Change_Switch']  = (df_stat[('GDP_%_Change', 'min')] * df_stat[('GDP_%_Change', 'max')]) < 0

In [ ]:
# List those quarters were an estimation of economic contraction has changed to growth or vice versa
df_stat[df_stat['GDP_%_Change_Switch'] == True]

In [ ]:
panel.head(10)

In [ ]:
panel[((panel['Quarter_Date'] >= '2025-01-01') & (panel['Quarter_Date'] <= '2026-01-01'))]